# Merge the Kaggle SFT v2 adapter into a standalone model (Colab)

Takes `sft_v2_best_adapter.zip` from the Kaggle run, checks it, and writes the merged model to `MyDrive/ahmed_twin/model_merged_nilechat_v2`, the folder the Colab SFT v2 notebook would have written. v1's `model_merged_nilechat` is never touched. About 10 minutes on a free T4.

1. Upload `sft_v2_best_adapter.zip` to `MyDrive/ahmed_twin/`.
2. Runtime → Change runtime type → T4 GPU, then Runtime → Run all. Approve Drive and paste your Hugging Face token.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import unsloth  # must come before transformers / peft
from unsloth import FastLanguageModel

import json
import os
import shutil
import zipfile
from getpass import getpass

from huggingface_hub import login

login(token=getpass("Hugging Face token: "))

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/ahmed_twin"
ZIP_PATH = f"{PROJECT_DIR}/sft_v2_best_adapter.zip"
LOCAL_DIR = "/content/sft_v2_adapter"  # unzipped on the Colab disk, not on Drive
MERGED_MODEL_PATH = f"{PROJECT_DIR}/model_merged_nilechat_v2"
V1_MERGED_MODEL_PATH = f"{PROJECT_DIR}/model_merged_nilechat"
assert MERGED_MODEL_PATH != V1_MERGED_MODEL_PATH, "v2 must never overwrite the v1 model"

MAX_SEQ_LENGTH = 2048
MIN_MERGED_MODEL_GB = 3  # a fp16 3B model is ~6 GB; less means a failed save
OVERWRITE_MERGED = False

In [ ]:
assert os.path.exists(ZIP_PATH), f"Missing {ZIP_PATH} -- upload sft_v2_best_adapter.zip from the Kaggle Output tab."
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(LOCAL_DIR)

ADAPTER_DIR = f"{LOCAL_DIR}/best_adapter"
SUMMARY_PATH = f"{LOCAL_DIR}/run_summary.json"
assert os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"), "The zip has no best_adapter/adapter_config.json."
with open(SUMMARY_PATH, encoding="utf-8") as f:
    summary = json.load(f)
assert summary["config"]["loss_on"] == "final_reply_only", "This adapter is not from the SFT v2 run."
print(f"Adapter from {summary['model']}: best eval loss {summary['best_eval_loss']:.4f}, "
      f"stopped at step {summary['stopped_at_step']} of {summary['planned_steps']}.")

In [ ]:
def verify_merged(path):
    # Check real file sizes, not just that the folder exists.
    files = os.listdir(path)
    total_gb = sum(os.path.getsize(os.path.join(path, f)) for f in files) / 1e9
    weights = [f for f in files if f.endswith(".safetensors") and "index" not in f]
    print(f"{path}: {total_gb:.2f} GB, weight shards: {weights}")
    assert total_gb > MIN_MERGED_MODEL_GB, f"Only {total_gb:.2f} GB -- the save failed (disk, quota, or interrupted)."
    assert weights, "No .safetensors weight shards -- only an index/config was saved."


if os.path.exists(MERGED_MODEL_PATH) and not OVERWRITE_MERGED:
    raise FileExistsError(f"{MERGED_MODEL_PATH} exists. Set OVERWRITE_MERGED = True to replace it.")

model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False)
model.save_pretrained_merged(MERGED_MODEL_PATH, tokenizer, save_method="merged_16bit")
verify_merged(MERGED_MODEL_PATH)
shutil.copy(SUMMARY_PATH, os.path.join(MERGED_MODEL_PATH, "sft_run_summary.json"))
print("Merged model ready:", MERGED_MODEL_PATH)